<a href="https://colab.research.google.com/github/JaysonGaa/cobble-llm/blob/main/cobble_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U huggingface_hub
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U trl
!pip install -q -U accelerate
!pip install -q -U bitsandbytes
!pip install -q -U datasets

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model has been loaded successfully")


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('cobble_token'))

print("Login Successful")

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded in 4-bit successfully")

In [ ]:
# Test base model
inputs = tokenizer("Fix this Python bug:\ndef add(a, b)\n    return a + b", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from datasets import load_dataset

# First Dataset are coding instruction pairs
dataset1 = load_dataset("sahil2801/CodeAlpaca-20k")

# Second Dataset has coding instructions
dataset2 = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K")

print(dataset1)
print(dataset2)

In [ ]:
# Look at a few examples from each dataset
print("=== CodeAlpaca Example ===")
print(dataset1['train'][0])

print("\n=== Magicoder Example ===")
print(dataset2['train'][0])

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K")

print(dataset)
print("\n=== Example 1 ===")
print(dataset['train'][0])

print("\n=== Example 2 ===")
print(dataset['train'][1])

print("\n=== Example 3 ===")
print(dataset['train'][2])

In [ ]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['problem']}

### Response:
{example['solution']}"""
    }

# Apply the formatting to the full dataset
formatted_dataset = dataset['train'].map(format_prompt)

# Drop columns we don't need
formatted_dataset = formatted_dataset.remove_columns(
    ['lang', 'raw_index', 'index', 'seed', 'openai_fingerprint', 'problem', 'solution']
)

print(formatted_dataset)
print("\n=== Formatted Example ===")
print(formatted_dataset[0]['text'])

In [ ]:
# Split into 95% train, 5% validation
split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

In [ ]:
# Save to disk so you can reload it instantly next session
train_dataset.save_to_disk("/content/cobble_train")
val_dataset.save_to_disk("/content/cobble_val")

print("Datasets saved successfully")
print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")

In [ ]:
from datasets import load_from_disk

train_dataset = load_from_disk("/content/cobble_train")
val_dataset = load_from_disk("/content/cobble_val")

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded successfully")

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./cobble-output",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    report_to="none"
)

print("Training arguments configured")

In [ ]:
from trl import SFTTrainer, SFTConfig

# CHANGE THESE 4 VALUES EACH SESSION
chunk_start = 2000
chunk_end = 5000

# First session
# checkpoint_to_load = None
# checkpoint_to_save = "./cobble-checkpoint-2k"

# Second Session
checkpoint_to_load = "./cobble-checkpoint-2k"
checkpoint_to_save = "./cobble-checkpoint-5k"


# Select chunk
train_chunk = train_dataset.select(range(chunk_start, chunk_end))
val_chunk = val_dataset.select(range(200))

print(f"Training on examples {chunk_start} to {chunk_end}")

trainer = SFTTrainer(
    model=model,
    train_dataset=train_chunk,
    eval_dataset=val_chunk,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir="./cobble-output",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
        logging_steps=25,
        learning_rate=2e-4,
        bf16=True,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_steps=50,
        report_to="none",
        dataset_text_field="text",
        max_length=512,
    ),
)

print("Starting training...")
trainer.train()

# Save checkpoint
trainer.model.save_pretrained(checkpoint_to_save)
tokenizer.save_pretrained(checkpoint_to_save)
print(f"Saved to {checkpoint_to_save}")

# Push to Hugging Face so you don't lose it
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=checkpoint_to_save,
    repo_id="jayson-gaa/Cobble-1B",
    repo_type="model"
)
print("Pushed to Hugging Face")